# Forcasting with foundational models

## Libraries

In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

/home/joaquin/Documents/GitHub/skforecast


0.21.0


In [2]:
# Libraries
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from skforecast.datasets import fetch_dataset
from skforecast.preprocessing import (
    reshape_series_wide_to_long,
    reshape_series_long_to_dict, 
)
from skforecast.foundational import ForecasterFoundational, FoundationalModel
from skforecast.model_selection import (
    TimeSeriesFold,
    backtesting_foundational
)
from skforecast.plot import set_dark_theme

In [3]:
# Data manipulation
# ==============================================================================
import numpy as np
import pandas as pd
from astral.sun import sun
from astral import LocationInfo
from skforecast.datasets import fetch_dataset
from feature_engine.datetime import DatetimeFeatures
from feature_engine.creation import CyclicalFeatures
from feature_engine.timeseries.forecasting import WindowFeatures

# Plots
# ==============================================================================
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from skforecast.plot import plot_residuals
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as poff
pio.templates.default = "seaborn"
poff.init_notebook_mode(connected=True)
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'font.size': 8})

# Modelling and Forecasting
# ==============================================================================
import skforecast
import lightgbm
import sklearn
from lightgbm import LGBMRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_selection import RFECV
from skforecast.recursive import ForecasterEquivalentDate, ForecasterRecursive
from skforecast.direct import ForecasterDirect
from skforecast.model_selection import TimeSeriesFold, bayesian_search_forecaster, backtesting_forecaster
from skforecast.feature_selection import select_features
from skforecast.preprocessing import RollingFeatures
from skforecast.plot import calculate_lag_autocorrelation, plot_residuals
from skforecast.metrics import calculate_coverage
import shap


# Warnings configuration
# ==============================================================================
import warnings
warnings.filterwarnings('once')

color = '\033[1m\033[38;5;208m' 
print(f"{color}Version skforecast: {skforecast.__version__}")
print(f"{color}Version scikit-learn: {sklearn.__version__}")
print(f"{color}Version lightgbm: {lightgbm.__version__}")
print(f"{color}Version pandas: {pd.__version__}")
print(f"{color}Version numpy: {np.__version__}")


Version skforecast: 0.21.0
Version scikit-learn: 1.8.0
Version lightgbm: 4.6.0
Version pandas: 2.3.3
Version numpy: 2.4.2


## Input data

In [4]:
# Data download
# ==============================================================================
data = fetch_dataset(name='vic_electricity', raw=True)
data.info()

# Data preparation
# ==============================================================================
data = data.copy()
data['Time'] = pd.to_datetime(data['Time'], format='%Y-%m-%dT%H:%M:%SZ')
data = data.set_index('Time')
data = data.asfreq('30min')
data = data.sort_index()
data.head(2)

# Aggregating in 1H intervals
# ==============================================================================
# The Date column is eliminated so that it does not generate an error when aggregating.
data = data.drop(columns="Date")
data = (
    data
    .resample(rule="h", closed="left", label="right")
    .agg({
        "Demand": "mean",
        "Temperature": "mean",
        "Holiday": "mean",
    })
)
data

# Split data into train-val-test
# ==============================================================================
data = data.loc['2012-01-01 00:00:00':'2014-12-30 23:00:00', :].copy()
end_train = '2013-12-31 23:59:00'
end_validation = '2014-11-30 23:59:00'
data_train = data.loc[: end_train, :].copy()
data_val   = data.loc[end_train:end_validation, :].copy()
data_test  = data.loc[end_validation:, :].copy()

print(f"Train dates      : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})")
print(f"Validation dates : {data_val.index.min()} --- {data_val.index.max()}  (n={len(data_val)})")
print(f"Test dates       : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})")


╭──────────────────────────── vic_electricity ─────────────────────────────╮
│ Description:                                                             │
│ Half-hourly electricity demand for Victoria, Australia                   │
│                                                                          │
│ Source:                                                                  │
│ O'Hara-Wild M, Hyndman R, Wang E, Godahewa R (2022).tsibbledata: Diverse │
│ Datasets for 'tsibble'. https://tsibbledata.tidyverts.org/,              │
│ https://github.com/tidyverts/tsibbledata/.                               │
│ https://tsibbledata.tidyverts.org/reference/vic_elec.html                │
│                                                                          │
│ URL:                                                                     │
│ https://raw.githubusercontent.com/skforecast/skforecast-                 │
│ datasets/main/data/vic_electricity.csv                                   │
│                                                                          │
│ Shape: 52608 rows x 5 columns                                            │
╰──────────────────────────────────────────────────────────────────────────╯

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52608 entries, 0 to 52607
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Time         52608 non-null  object 
 1   Demand       52608 non-null  float64
 2   Temperature  52608 non-null  float64
 3   Date         52608 non-null  object 
 4   Holiday      52608 non-null  bool   
dtypes: bool(1), float64(2), object(2)
memory usage: 1.7+ MB
Train dates      : 2012-01-01 00:00:00 --- 2013-12-31 23:00:00  (n=17544)
Validation dates : 2014-01-01 00:00:00 --- 2014-11-30 23:00:00  (n=8016)
Test dates       : 2014-12-01 00:00:00 --- 2014-12-30 23:00:00  (n=720)


## ForecasterFoundational

In [5]:
# Create and train ForecasterFoundational
# ==============================================================================
estimator = FoundationalModel(model="autogluon/chronos-2-small")
forecaster = ForecasterFoundational(estimator = estimator)
forecaster.fit(series=data_train["Demand"], exog=data_train[["Temperature", "Holiday"]])
forecaster

====================== 
ForecasterFoundational 
====================== 
Model: autogluon/chronos-2-small 
Context length: 2048 
Series name: Demand 
Exogenous included: True 
Exogenous names: Temperature, Holiday 
Training range: [Timestamp('2012-01-01 00:00:00'), Timestamp('2013-12-31 23:00:00')] 
Training index type: DatetimeIndex 
Training index frequency: <Hour> 
Creation date: 2026-03-26 21:34:15 
Last fit date: 2026-03-26 21:34:15 
Skforecast version: 0.21.0 
Python version: 3.13.11 
Forecaster id: None

Two methods can be used to predict the next n steps: `predict()` or `predict_interval()`. The argument `levels` is used to indicate for which series estimate predictions. If `None` all series will be predicted.

In [6]:
# Predictions and prediction intervals
# ==============================================================================
steps = 24

# Predictions for item_1
predictions_item_1 = forecaster.predict(steps=steps)
display(predictions_item_1.head(3))
print("")

# Interval predictions for item_1 and item_2
predictions_intervals = forecaster.predict_interval(
    steps    = steps,
    interval = 0.9
)
display(predictions_intervals)

2014-01-01 00:00:00    3626.093262
2014-01-01 01:00:00    3703.165527
2014-01-01 02:00:00    3757.089355
Freq: h, Name: pred, dtype: float32

,pred,lower_bound,upper_bound
2014-01-01 00:00:00,3626.093262,3489.707031,3746.984375
2014-01-01 01:00:00,3703.165527,3522.691895,3882.696777
2014-01-01 02:00:00,3757.089355,3551.117920,3972.982666
2014-01-01 03:00:00,3785.650879,3532.992676,4042.825439
2014-01-01 04:00:00,3819.878174,3547.810547,4113.866699
2014-01-01 05:00:00,3941.915039,3615.704102,4236.655762
2014-01-01 06:00:00,4084.348389,3755.987061,4410.641602
2014-01-01 07:00:00,4185.162598,3830.577148,4517.246094
2014-01-01 08:00:00,4126.264648,3787.660400,4440.847168
2014-01-01 09:00:00,3989.280029,3703.944092,4281.961426


## Backtesting multiple series

In [7]:
# Backtesting multiple time series
# ==============================================================================
cv = TimeSeriesFold(
        steps              = 24,
        initial_train_size = len(data.loc[:end_validation]),
        refit              = False
)

metrics_levels, backtest_predictions = backtesting_foundational(
    forecaster            = forecaster,
    series                = data['Demand'],
    exog                  = data[["Temperature", "Holiday"]],
    cv                    = cv,
    metric                = 'mean_absolute_error',
    add_aggregated_metric = True
)

print("Backtest metrics")
display(metrics_levels)
print("")
print("Backtest predictions")
backtest_predictions.head(4)


  0%|          | 0/30 [00:00<?, ?it/s]

Backtest metrics


,mean_absolute_error
0,157.032158



Backtest predictions


,fold,pred
Time,,
2014-12-01 00:00:00,0,5501.366699
2014-12-01 01:00:00,0,5450.031738
2014-12-01 02:00:00,0,5375.812012
2014-12-01 03:00:00,0,5315.422852


In [8]:
# Plot predictions vs real value
# ======================================================================================
fig = go.Figure()
trace1 = go.Scatter(x=data_test.index, y=data_test['Demand'], name="test", mode="lines")
trace2 = go.Scatter(x=backtest_predictions.index, y=backtest_predictions['pred'], name="prediction", mode="lines")
fig.add_trace(trace1)
fig.add_trace(trace2)
fig.update_layout(
    title="Real value vs predicted in test data",
    xaxis_title="Date time",
    yaxis_title="Demand",
    width=800,
    height=400,
    margin=dict(l=20, r=20, t=35, b=20),
    legend=dict(orientation="h", yanchor="top", y=1.01, xanchor="left", x=0)
)
fig.show()